In [ ]:
# ============================================
# ONE-CELL ADAPTIVE LOOKBACK FULL RUN ON ALL 9 DATASETS
# Run this in a NEW notebook: 09_adaptive_lookback_fullrun.ipynb
# ============================================

from pathlib import Path
import sys
import importlib
import gc
import json
import math
import random
import copy
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------------------------------------
# 0) Project setup
# --------------------------------------------
ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project").resolve()
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner file: {RUNNER_PATH}")

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

OUT_TABLES = ROOT / "results" / "tables" / "phase5"
OUT_PREDS = ROOT / "results" / "predictions" / "phase5_adaptive_lookback"
OUT_CKPTS = ROOT / "results" / "checkpoints" / "phase5_adaptive_lookback"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_PREDS.mkdir(parents=True, exist_ok=True)
OUT_CKPTS.mkdir(parents=True, exist_ok=True)

MASTER_PATH = ROOT / "results" / "tables" / "master_all_models_metrics_unified.csv"
BEST_PATH = ROOT / "results" / "tables" / "master_best_by_dataset_unified.csv"

# --------------------------------------------
# 1) Hardware / efficiency setup
# --------------------------------------------
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
else:
    GPU_NAME = "CPU"
    GPU_MEM_GB = 0.0

USE_AMP = bool(torch.cuda.is_available()) and bool(getattr(p5, "TRAIN_CFG", {}).get("use_amp", True))
SEED = 42
PATIENCE = int(getattr(p5, "TRAIN_CFG", {}).get("full_patience", 8))

print("Imported p5 from:", RUNNER_PATH)
print("DEVICE:", p5.DEVICE)
print("GPU_NAME:", GPU_NAME)
print("GPU_MEM_GB:", round(GPU_MEM_GB, 2))
print("USE_AMP:", USE_AMP)
print("PATIENCE:", PATIENCE)

# --------------------------------------------
# 2) Fixed base architecture
# --------------------------------------------
ALL_DATASETS = list(p5.ALL_DATASETS)

BASE_ARCH = {
    "levels": 2,
    "wavelet_family": "Db4",
    "d_model": 128,
    "num_backbone_blocks": 2,
    "dropout": 0.10,
    "filter_reg_lambda": 1e-4,
    "gate_entropy_lambda": 1e-4,
}

# Adaptive-lookback config
# This learns a soft mixture over candidate recent-history windows.
LOOKBACK_CFG = {
    "temperature": 1.0,
    "lambda_entropy": 1e-3,     # encourage decisive selection
    "lambda_shortness": 5e-4,   # mild preference against always using the full history
    "hidden_cap": 128,
}

# GPU-aware batch multiplier
if GPU_MEM_GB >= 80:
    BATCH_MUL = {
        "etth1": 2,
        "etth2": 2,
        "ettm1": 2,
        "ettm2": 2,
        "weather": 2,
        "electricity": 1,
        "traffic": 1,
        "exchange": 2,
        "ili": 2,
    }
else:
    BATCH_MUL = {ds: 1 for ds in ALL_DATASETS}

print("ALL_DATASETS:", ALL_DATASETS)
print("BASE_ARCH:", BASE_ARCH)
print("LOOKBACK_CFG:", LOOKBACK_CFG)
print("BATCH_MUL:", BATCH_MUL)

# --------------------------------------------
# 3) Helpers
# --------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))

def resolve_seq_len(ds):
    seq_candidates = p5.resolve_seq_candidates(ds, "searched")
    return int(seq_candidates[min(1, len(seq_candidates) - 1)])

def resolve_batch_size(ds):
    base = int(p5.DEFAULT_BATCH_SIZE[ds])
    return int(base * BATCH_MUL[ds])

def resolve_epochs(ds):
    return int(p5.DEFAULT_EPOCHS[ds])

def build_candidate_lookbacks(dataset_name, seq_len):
    # Use dataset-specific searched candidates where possible, then add fraction-based windows.
    cands = []
    try:
        for c in p5.resolve_seq_candidates(dataset_name, "searched"):
            c = int(c)
            if 4 <= c <= seq_len:
                cands.append(c)
    except Exception:
        pass

    for frac in [0.25, 0.50, 0.75, 1.00]:
        c = max(4, int(round(seq_len * frac)))
        c = min(seq_len, c)
        cands.append(c)

    cands = sorted(set(cands))
    if seq_len not in cands:
        cands.append(seq_len)
    return cands

def metric_row_from_npz(dataset, model, family, seq_len, pred_len, pred_file_rel):
    p = ROOT / str(pred_file_rel)
    arr = np.load(p, allow_pickle=True)
    preds = np.asarray(arr["preds"], dtype=np.float64).reshape(-1)
    trues = np.asarray(arr["trues"], dtype=np.float64).reshape(-1)
    return {
        "dataset": dataset,
        "model": model,
        "family": family,
        "seq_len": int(seq_len),
        "pred_len": int(pred_len),
        "test_mse": mse(trues, preds),
        "test_mae": mae(trues, preds),
        "test_rmse": rmse(trues, preds),
        "prediction_file": str(pred_file_rel),
    }

# --------------------------------------------
# 4) Adaptive lookback wrapper
# --------------------------------------------
class AdaptiveLookbackWrapper(nn.Module):
    """
    Learns a soft mixture over candidate recent-history windows.
    It gates the input sequence before feeding it to the base wavelet model.
    """
    def __init__(self, base_model, seq_len, input_dim, candidate_lengths, temperature=1.0, dropout=0.05, hidden_cap=128):
        super().__init__()
        self.base_model = base_model
        self.seq_len = int(seq_len)
        self.input_dim = int(input_dim)
        self.temperature = float(temperature)

        candidate_lengths = [int(x) for x in candidate_lengths]
        self.register_buffer("candidate_lengths", torch.tensor(candidate_lengths, dtype=torch.long))

        masks = []
        for L in candidate_lengths:
            m = torch.zeros(self.seq_len, dtype=torch.float32)
            m[-int(L):] = 1.0
            masks.append(m)
        self.register_buffer("candidate_masks", torch.stack(masks, dim=0))  # [K, T]

        hidden = min(hidden_cap, max(32, self.seq_len // 2))
        self.selector = nn.Sequential(
            nn.LayerNorm(self.seq_len),
            nn.Linear(self.seq_len, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, len(candidate_lengths)),
        )

    def forward(self, x):
        # x: [B, T, C]
        # summary signal for lookback selection
        profile = x.abs().mean(dim=2)  # [B, T]
        logits = self.selector(profile)  # [B, K]
        weights = torch.softmax(logits / self.temperature, dim=-1)  # [B, K]

        soft_mask = weights @ self.candidate_masks  # [B, T]
        x_masked = x * soft_mask.unsqueeze(-1)

        pred, aux = self.base_model(x_masked)
        if aux is None or not isinstance(aux, dict):
            aux = {}
        aux["lookback_weights"] = weights
        aux["lookback_mask"] = soft_mask
        aux["candidate_lengths"] = self.candidate_lengths
        return pred, aux

def adaptive_lookback_loss(pred_scaled, y_scaled, aux, cfg, seq_len):
    base = F.mse_loss(pred_scaled, y_scaled)

    lookback_weights = aux["lookback_weights"]
    lengths = aux["candidate_lengths"].float().to(lookback_weights.device)

    # entropy penalty: encourage sharper choices
    entropy = -(lookback_weights * (lookback_weights.clamp_min(1e-8).log())).sum(dim=-1).mean()

    # mild expected-length penalty: avoid always collapsing to full history
    expected_len = (lookback_weights * lengths.unsqueeze(0)).sum(dim=-1).mean()
    expected_frac = expected_len / float(seq_len)

    total = base + cfg["lambda_entropy"] * entropy + cfg["lambda_shortness"] * expected_frac

    parts = {
        "base_mse": float(base.detach().cpu().item()),
        "entropy": float(entropy.detach().cpu().item()),
        "expected_frac": float(expected_frac.detach().cpu().item()),
    }
    return total, parts

@torch.no_grad()
def evaluate_model(model, loader, target_mean, target_std):
    model.eval()
    preds = []
    trues = []
    weight_rows = []

    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred_scaled, aux = model(x)

        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)

        if "lookback_weights" in aux:
            weight_rows.append(aux["lookback_weights"].detach().cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)

    avg_weights = None
    if len(weight_rows) > 0:
        avg_weights = np.concatenate(weight_rows, axis=0).mean(axis=0)

    return {
        "preds": preds,
        "trues": trues,
        "mse": mse(trues.reshape(-1), preds.reshape(-1)),
        "mae": mae(trues.reshape(-1), preds.reshape(-1)),
        "rmse": rmse(trues.reshape(-1), preds.reshape(-1)),
        "avg_lookback_weights": avg_weights,
    }

# --------------------------------------------
# 5) One full dataset run
# --------------------------------------------
def train_one_dataset(dataset_name):
    set_seed(SEED)

    bundle = p5.load_bundle(dataset_name, input_mode="multivariate")
    pred_len = p5.resolve_pred_len(dataset_name, "long")
    seq_len = resolve_seq_len(dataset_name)
    batch_size = resolve_batch_size(dataset_name)
    epochs = resolve_epochs(dataset_name)

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, batch_size)

    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    base_model = p5.AdaptiveWaveletMixer(
        input_dim=int(bundle["values_scaled"].shape[1]),
        pred_len=pred_len,
        d_model=int(BASE_ARCH["d_model"]),
        levels=int(BASE_ARCH["levels"]),
        wavelet_family=BASE_ARCH["wavelet_family"],
        num_backbone_blocks=int(BASE_ARCH["num_backbone_blocks"]),
        dropout=float(BASE_ARCH["dropout"]),
        filter_reg_lambda=float(BASE_ARCH["filter_reg_lambda"]),
        gate_entropy_lambda=float(BASE_ARCH["gate_entropy_lambda"]),
    )

    candidate_lengths = build_candidate_lookbacks(dataset_name, seq_len)

    model = AdaptiveLookbackWrapper(
        base_model=base_model,
        seq_len=seq_len,
        input_dim=int(bundle["values_scaled"].shape[1]),
        candidate_lengths=candidate_lengths,
        temperature=LOOKBACK_CFG["temperature"],
        dropout=BASE_ARCH["dropout"],
        hidden_cap=LOOKBACK_CFG["hidden_cap"],
    ).to(p5.DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5,
        foreach=False,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_rmse = float("inf")
    best_epoch = -1
    best_state = None
    best_val_weights = None
    patience_left = PATIENCE

    history = []
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        train_base = []
        train_entropy = []
        train_expfrac = []

        for batch in train_loader:
            x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred_scaled, aux = model(x)
                loss, parts = adaptive_lookback_loss(pred_scaled, y, aux, LOOKBACK_CFG, seq_len)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_losses.append(float(loss.detach().cpu().item()))
            train_base.append(parts["base_mse"])
            train_entropy.append(parts["entropy"])
            train_expfrac.append(parts["expected_frac"])

        val_eval = evaluate_model(model, val_loader, target_mean, target_std)
        val_rmse = float(val_eval["rmse"])
        mean_train_loss = float(np.mean(train_losses)) if len(train_losses) > 0 else np.nan

        history.append({
            "dataset": dataset_name,
            "epoch": epoch,
            "train_loss": mean_train_loss,
            "train_base_mse": float(np.mean(train_base)) if len(train_base) > 0 else np.nan,
            "train_entropy": float(np.mean(train_entropy)) if len(train_entropy) > 0 else np.nan,
            "train_expected_frac": float(np.mean(train_expfrac)) if len(train_expfrac) > 0 else np.nan,
            "val_rmse": val_rmse,
            "val_mae": float(val_eval["mae"]),
            "val_mse": float(val_eval["mse"]),
            "candidate_lengths_json": json.dumps(candidate_lengths),
            "avg_val_lookback_weights_json": json.dumps(
                val_eval["avg_lookback_weights"].tolist() if val_eval["avg_lookback_weights"] is not None else []
            ),
        })

        print(
            f"[{dataset_name}] epoch {epoch:03d} | "
            f"train_loss={mean_train_loss:.6f} | val_rmse={val_rmse:.6f} | "
            f"batch_size={batch_size} | candidates={candidate_lengths}"
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            best_val_weights = None if val_eval["avg_lookback_weights"] is None else val_eval["avg_lookback_weights"].copy()
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{dataset_name}] early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)

    test_eval = evaluate_model(model, test_loader, target_mean, target_std)

    ckpt_path = OUT_CKPTS / f"{dataset_name}_AdaptiveWaveletMixer_AdaptiveLookback.pt"
    pred_path = OUT_PREDS / f"{dataset_name}_AdaptiveWaveletMixer_AdaptiveLookback_test_predictions.npz"

    torch.save({
        "dataset": dataset_name,
        "model": "AdaptiveWaveletMixer_AdaptiveLookback",
        "base_arch": BASE_ARCH,
        "lookback_cfg": LOOKBACK_CFG,
        "candidate_lengths": candidate_lengths,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "best_epoch": best_epoch,
        "state_dict": model.state_dict(),
    }, ckpt_path)

    np.savez_compressed(
        pred_path,
        preds=test_eval["preds"],
        trues=test_eval["trues"],
        seq_len=seq_len,
        pred_len=pred_len,
        candidate_lengths=np.array(candidate_lengths, dtype=np.int64),
        avg_test_lookback_weights=np.array(test_eval["avg_lookback_weights"]) if test_eval["avg_lookback_weights"] is not None else np.array([]),
    )

    row = {
        "dataset": dataset_name,
        "model": "AdaptiveWaveletMixer_AdaptiveLookback",
        "family": "wavelet_adaptive",
        "use_adaptive_lookback": True,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "batch_size": batch_size,
        "epochs_target": epochs,
        "best_epoch": int(best_epoch),
        "train_seconds": float(time.time() - t0),
        "candidate_lengths_json": json.dumps(candidate_lengths),
        "avg_best_val_lookback_weights_json": json.dumps(best_val_weights.tolist() if best_val_weights is not None else []),
        "avg_test_lookback_weights_json": json.dumps(
            test_eval["avg_lookback_weights"].tolist() if test_eval["avg_lookback_weights"] is not None else []
        ),
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(test_eval["mse"]),
        "test_mae": float(test_eval["mae"]),
        "test_rmse": float(test_eval["rmse"]),
    }

    hist_df = pd.DataFrame(history)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row, hist_df

# --------------------------------------------
# 6) Run all 9 datasets
# --------------------------------------------
all_rows = []
all_hist = []

print("\nStarting full Adaptive Lookback run on all 9 datasets...\n")

for ds in ALL_DATASETS:
    print("\n" + "=" * 120)
    print(f"RUNNING ADAPTIVE LOOKBACK | dataset={ds}")
    print("=" * 120)
    row, hist_df = train_one_dataset(ds)
    all_rows.append(row)
    all_hist.append(hist_df)

full_df = pd.DataFrame(all_rows)
hist_df = pd.concat(all_hist, ignore_index=True)

metrics_csv = OUT_TABLES / "phase5_adaptive_lookback_full_metrics.csv"
history_csv = OUT_TABLES / "phase5_adaptive_lookback_full_history.csv"
registry_csv = OUT_TABLES / "phase5_adaptive_lookback_full_prediction_registry.csv"

full_df.to_csv(metrics_csv, index=False)
hist_df.to_csv(history_csv, index=False)
full_df[[
    "dataset", "model", "family", "seq_len", "pred_len",
    "checkpoint", "prediction_file", "test_mse", "test_mae", "test_rmse",
    "candidate_lengths_json", "avg_test_lookback_weights_json"
]].to_csv(registry_csv, index=False)

print("\nSaved full metrics:", metrics_csv)
print("Saved history:", history_csv)
print("Saved prediction registry:", registry_csv)
display(full_df)

# --------------------------------------------
# 7) Build merged candidate master tables
# --------------------------------------------
if not MASTER_PATH.exists():
    raise FileNotFoundError(f"Missing master file: {MASTER_PATH}")
if not BEST_PATH.exists():
    raise FileNotFoundError(f"Missing best file: {BEST_PATH}")

master = pd.read_csv(MASTER_PATH)
best_old = pd.read_csv(BEST_PATH)

master_candidate = master[master["model"] != "AdaptiveWaveletMixer_AdaptiveLookback"].copy()

new_rows = []
for _, r in full_df.iterrows():
    new_rows.append(
        metric_row_from_npz(
            dataset=r["dataset"],
            model=r["model"],
            family=r["family"],
            seq_len=r["seq_len"],
            pred_len=r["pred_len"],
            pred_file_rel=r["prediction_file"],
        )
    )

new_df = pd.DataFrame(new_rows)
master_candidate = pd.concat([master_candidate, new_df], ignore_index=True)
master_candidate = master_candidate.sort_values(["dataset", "family", "model"]).reset_index(drop=True)

best_candidate = (
    master_candidate.sort_values(["dataset", "test_rmse", "test_mae", "test_mse"])
                    .groupby("dataset", as_index=False)
                    .first()[["dataset", "model", "family", "test_mse", "test_mae", "test_rmse"]]
)

candidate_master_csv = ROOT / "results" / "tables" / "master_all_models_metrics_unified_lookback_candidate.csv"
candidate_best_csv = ROOT / "results" / "tables" / "master_best_by_dataset_unified_lookback_candidate.csv"

master_candidate.to_csv(candidate_master_csv, index=False)
best_candidate.to_csv(candidate_best_csv, index=False)

print("\nSaved candidate merged master:", candidate_master_csv)
print("Saved candidate best table:", candidate_best_csv)

# --------------------------------------------
# 8) Compare adaptive lookback against current best
# --------------------------------------------
compare = best_old.rename(columns={
    "model": "old_best_model",
    "family": "old_best_family",
    "test_mse": "old_best_mse",
    "test_mae": "old_best_mae",
    "test_rmse": "old_best_rmse",
}).merge(
    full_df[[
        "dataset", "model", "test_mse", "test_mae", "test_rmse",
        "candidate_lengths_json", "avg_test_lookback_weights_json"
    ]].rename(columns={
        "model": "lookback_model",
        "test_mse": "lookback_mse",
        "test_mae": "lookback_mae",
        "test_rmse": "lookback_rmse",
    }),
    on="dataset",
    how="left"
)

compare["rmse_gain_abs"] = compare["old_best_rmse"] - compare["lookback_rmse"]
compare["rmse_gain_pct"] = 100.0 * compare["rmse_gain_abs"] / compare["old_best_rmse"]

compare["mae_gain_abs"] = compare["old_best_mae"] - compare["lookback_mae"]
compare["mae_gain_pct"] = 100.0 * compare["mae_gain_abs"] / compare["old_best_mae"]

compare_csv = OUT_TABLES / "phase5_adaptive_lookback_full_vs_current_best.csv"
compare.to_csv(compare_csv, index=False)

wins = int((compare["rmse_gain_abs"] > 0).sum())
losses = int((compare["rmse_gain_abs"] < 0).sum())
ties = int((compare["rmse_gain_abs"] == 0).sum())

print("\nSaved comparison table:", compare_csv)
display(compare)

print("\n" + "=" * 120)
print("FULL ADAPTIVE LOOKBACK DECISION")
print("=" * 120)
print(f"Adaptive lookback wins on {wins}/{len(ALL_DATASETS)} datasets by RMSE")
print(f"Adaptive lookback loses on {losses}/{len(ALL_DATASETS)} datasets by RMSE")
print(f"Ties: {ties}")

if wins >= 4:
    print("RECOMMENDATION: Adaptive lookback is promising enough to keep as the next main improved candidate.")
else:
    print("RECOMMENDATION: Adaptive lookback is not strong enough overall. Then we should redesign the mutation again instead of scaling more runs.")

print("\nCandidate best-by-dataset table:")
display(best_candidate)